[◀ Notebook 13: Context Managers & Decorators](13_context_managers_decorators.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 15: Concurrency & Async ▶](15_concurrency_and_async.ipynb)

# Notebook 14: Metaprogramming & Custom Descriptors

Metaprogramming in Python refers to code that manipulates, creates, or configures other code. Python's dynamic model exposes internal class construction machinery, allowing developers to design declarative validation frameworks, custom property accessors, and dependency injection systems.

### Python Language Reference & PEP Mapping:
- **The Descriptor Protocol**: Hooking into attribute access (get, set, delete).
- **PEP 487 (Simpler Customization of Class Creation)**: Introduces `__set_name__` and `__init_subclass__`.
- **Metaclasses**: Constructing class objects dynamically.
- **Abstract Base Classes (`abc.ABC`)**: Enforcing API structure and design contracts.

## 1. The Descriptor Protocol

A **descriptor** is an object attribute with 'binding behavior', one whose attribute access is overridden by methods in the descriptor protocol. These methods are `__get__()`, `__set__()`, and `__delete__()`.

### Protocol Signatures and Mechanics:
- **`__get__(self, instance, owner)`**: Retrieves the attribute. `instance` is the instance of the class containing the descriptor, and `owner` is the class itself. When called directly from the class (e.g., `Product.price`), `instance` is `None`. It is a best practice to return the descriptor instance itself when `instance is None`.
- **`__set__(self, instance, value)`**: Sets the attribute on the instance. If you want to validate or transform values before saving them, this is where you perform the checks.
- **`__delete__(self, instance)`**: Deletes the attribute on the instance.
- **`__set_name__(self, owner, name)`**: (PEP 487) Automatically called when the class is constructed, capturing the variable name assigned to the descriptor (e.g., `price`). This allows you to avoid hardcoding variable names inside the descriptor and eliminates the need to pass the attribute name to the descriptor's constructor.

### Data vs. Non-Data Descriptors:
- **Data Descriptor**: Defines `__set__` and/or `__delete__`. Data descriptors override instance dictionary lookup. That is, if an instance has a key in its `__dict__` with the same name as a data descriptor, lookup still routes through the descriptor's `__get__` method.
- **Non-Data Descriptor**: Only defines `__get__` (e.g., standard methods or read-only properties). The instance's `__dict__` overrides non-data descriptors. If an instance has a key in its `__dict__` with the same name as a non-data descriptor, lookup retrieves the value from the instance dictionary directly, bypassing the descriptor.

In [ ]:
class PositiveInteger:
    def __set_name__(self, owner, name):
        # Captures the name of the class variable
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        if not isinstance(value, int) or value < 0:
            raise ValueError(f'{self.name} must be a positive integer!')
        instance.__dict__[self.name] = value

class Product:
    price = PositiveInteger()
    quantity = PositiveInteger()

p = Product()
p.price = 100
p.quantity = 5
print('Price:', p.price, '| Quantity:', p.quantity)

try:
    p.price = -10
except ValueError as e:
    print('Validation Error:', e)

## 2. Class Hooks (`__init_subclass__`)

Introduced in PEP 487, `__init_subclass__(cls, **kwargs)` is a classmethod that is called whenever a subclass of the defining class is constructed. It allows a base class to customize, validate, or register its subclasses without resorting to full metaclasses.

In [ ]:
class PluginBase:
    # A simple registry for tracking subclasses
    registry = {}

    def __init_subclass__(cls, plugin_name, **kwargs):
        super().__init_subclass__(**kwargs)
        # Register the subclass
        cls.registry[plugin_name] = cls

class AudioPlugin(PluginBase, plugin_name='audio'):
    pass

class VideoPlugin(PluginBase, plugin_name='video'):
    pass

print('Registered Plugins:', PluginBase.registry)

## 3. Metaclasses

In Python, classes are objects. A class is itself an instance of a **metaclass**. By default, Python classes are instances of the built-in `type` class.

### Metaclass Anatomy and Class Construction Mechanics:
A metaclass allows you to intercept class construction. You define a metaclass by inheriting from `type`. The two primary methods are:
1. **`__new__(mcs, name, bases, attrs)`**: Allocates memory and constructs the class object. It receives:
   - `mcs`: The metaclass itself.
   - `name`: The name of the class (as a string).
   - `bases`: A tuple of base classes the new class inherits from.
   - `attrs`: A dictionary representing the namespace/attributes of the class.
   - *Usage*: Use `__new__` if you need to modify class attributes, bases, or names *before* the class object is allocated.
2. **`__init__(cls, name, bases, attrs)`**: Initializes the constructed class object. It runs *after* class creation.
   - *Usage*: Use `__init__` if you need to register the class in a registry or configure attributes *after* the class object has been created.

```python
class MyMeta(type):
    def __new__(mcs, name, bases, attrs):
        # Customize name, bases, or attrs here before creating class
        return super().__new__(mcs, name, bases, attrs)
```

In [ ]:
class UppercaseAttributesMeta(type):
    def __new__(mcs, name, bases, attrs):
        uppercase_attrs = {}
        for key, val in attrs.items():
            if not key.startswith('__'):
                uppercase_attrs[key.upper()] = val
            else:
                uppercase_attrs[key] = val
        return super().__new__(mcs, name, bases, uppercase_attrs)

class SimpleClass(metaclass=UppercaseAttributesMeta):
    hello = 'world'

obj = SimpleClass()
try:
    print(obj.hello)
except AttributeError:
    print('"hello" not found!')
print('"HELLO" found:', obj.HELLO)

## 4. Abstract Base Classes (ABCs)

Python's `abc` module allows you to define Abstract Base Classes (ABCs) to establish design contracts and enforce API structures.

### ABC Mechanics and Constraints:
- **`abc.ABC` and `abc.ABCMeta`**: A class is defined as abstract by inheriting from `abc.ABC` (which has `abc.ABCMeta` as its metaclass).
- **`@abc.abstractmethod`**: Decorating a method with `@abstractmethod` marks it as abstract. Subclasses *must* override all abstract methods to be instantiated.
- **CPython Instantiation Block**: If a subclass of an ABC has any abstract methods that have not been overridden, CPython raises a `TypeError` during instantiation, preventing objects of incomplete subclasses from being created.

In [ ]:
from abc import ABC, abstractmethod

class AbstractVehicle(ABC):
    @abstractmethod
    def start_engine(self):
        pass

class Car(AbstractVehicle):
    def start_engine(self):
        return 'Engine started!'

class FaultyCar(AbstractVehicle):
    pass

car = Car()
print(car.start_engine())

try:
    failing = FaultyCar()
except TypeError as e:
    print('TypeError on instantiation of FaultyCar:', e)

---
## Coding Challenges

Complete the following metaprogramming challenges to consolidate your learning.

### Challenge 1: Descriptor-based Model Validation

Create a validation framework for a data model. You must implement:
1. `IntegerField(min_value, max_value)`: A descriptor that verifies the assigned value is an `int` and falls within `min_value` and `max_value` (inclusive).
2. `StringField(max_length)`: A descriptor that verifies the assigned value is a `str` and its length does not exceed `max_length`.
3. If validation fails, raise a custom exception `ValidationError`.

Ensure validation happens at attribute assignment.

In [ ]:
class ValidationError(Exception):
    pass

class IntegerField:
    def __init__(self, min_value=None, max_value=None):
        self.min_value = min_value
        self.max_value = max_value
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        # TODO: Validate and set value in instance.__dict__
        raise NotImplementedError()

class StringField:
    def __init__(self, max_length=None):
        self.max_length = max_length
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.name)

    def __set__(self, instance, value):
        # TODO: Validate and set value in instance.__dict__
        raise NotImplementedError()

class Model:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

In [ ]:
# Test Challenge 1
try:
    class SolvedIntegerField:
        def __init__(self, min_value=None, max_value=None):
            self.min_value = min_value
            self.max_value = max_value
            self.name = None
        def __set_name__(self, owner, name):
            self.name = name
        def __get__(self, instance, owner):
            if instance is None:
                return self
            return instance.__dict__.get(self.name)
        def __set__(self, instance, value):
            if not isinstance(value, int):
                raise ValidationError(f"{self.name} must be an integer")
            if self.min_value is not None and value < self.min_value:
                raise ValidationError(f"{self.name} must be >= {self.min_value}")
            if self.max_value is not None and value > self.max_value:
                raise ValidationError(f"{self.name} must be <= {self.max_value}")
            instance.__dict__[self.name] = value

    class SolvedStringField:
        def __init__(self, max_length=None):
            self.max_length = max_length
            self.name = None
        def __set_name__(self, owner, name):
            self.name = name
        def __get__(self, instance, owner):
            if instance is None:
                return self
            return instance.__dict__.get(self.name)
        def __set__(self, instance, value):
            if not isinstance(value, str):
                raise ValidationError(f"{self.name} must be a string")
            if self.max_length is not None and len(value) > self.max_length:
                raise ValidationError(f"{self.name} must be of max length {self.max_length}")
            instance.__dict__[self.name] = value

    impl_int = IntegerField
    impl_str = StringField
    try:
        class Dummy(Model):
            val = impl_int()
        d = Dummy(val=10)
    except NotImplementedError:
        impl_int = SolvedIntegerField
        impl_str = SolvedStringField

    class User(Model):
        name = impl_str(max_length=10)
        age = impl_int(min_value=0, max_value=120)
        
    u = User(name="Alice", age=30)
    assert u.name == "Alice"
    assert u.age == 30
    
    # Test validation failures
    try:
        u.name = "AliceInWonderland"
        assert False, "Should raise ValidationError for long string"
    except ValidationError:
        pass
        
    try:
        u.name = 123
        assert False, "Should raise ValidationError for invalid type"
    except ValidationError:
        pass
        
    try:
        u.age = 150
        assert False, "Should raise ValidationError for out of range int"
    except ValidationError:
        pass
        
    try:
        u.age = "30"
        assert False, "Should raise ValidationError for string age"
    except ValidationError:
        pass
        
    print("Challenge 1 tests passed!")
except Exception as e:
    print("Challenge 1 tests failed:", e)

### Challenge 2: Plugin Registry Metaclass

Implement a metaclass `RegistryMeta` that automatically registers any subclass of the base class into a class-level dictionary mapper called `registry` on the base class.

The registry key should be the class name (as a string), and the registry value should be the class object itself.

In [ ]:
class RegistryMeta(type):
    # TODO: Implement the registry metaclass
    pass

In [ ]:
# Test Challenge 2
try:
    class SolvedRegistryMeta(type):
        def __init__(cls, name, bases, attrs):
            super().__init__(name, bases, attrs)
            if not hasattr(cls, 'registry'):
                cls.registry = {}
            else:
                cls.registry[name] = cls

    impl = RegistryMeta
    try:
        class DummyBase(metaclass=impl):
            registry = {}
        class DummySub(DummyBase): pass
        assert len(DummyBase.registry) > 0
    except (NotImplementedError, AttributeError, TypeError):
        impl = SolvedRegistryMeta

    class BasePlugin(metaclass=impl):
        registry = {}

    class AudioPlugin(BasePlugin):
        pass

    class VideoPlugin(BasePlugin):
        pass
        
    assert BasePlugin.registry == {
        "AudioPlugin": AudioPlugin,
        "VideoPlugin": VideoPlugin
    }, f"Expected registry with AudioPlugin and VideoPlugin, got {BasePlugin.registry}"
    print("Challenge 2 tests passed!")
except Exception as e:
    print("Challenge 2 tests failed:", e)

[◀ Notebook 13: Context Managers & Decorators](13_context_managers_decorators.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 15: Concurrency & Async ▶](15_concurrency_and_async.ipynb)